# 01 — Data Collection

This notebook walks through pulling Reddit posts, TMDB metadata, and box office data for each film.

**Skip this if you just want to run the dashboard** — `data/sample_data.csv` works without any API keys.


In [ ]:
import sys
sys.path.append('..')
import pandas as pd
from src.reddit_collector import collect_pre_release_posts, compute_engagement_features
from dotenv import load_dotenv
load_dotenv('../.env')


## Define your film list

Start small — 10 films. Once the pipeline is validated, scale to 100+.

In [ ]:
FILMS_TO_COLLECT = [
    {"title": "Oppenheimer", "release_date": "2023-07-21"},
    {"title": "Barbie", "release_date": "2023-07-21"},
    {"title": "The Flash", "release_date": "2023-06-16"},
    {"title": "Top Gun Maverick", "release_date": "2022-05-27"},
    {"title": "Tenet", "release_date": "2020-09-03"},
]


In [ ]:
all_posts = {}
engagement_features = {}

for film in FILMS_TO_COLLECT:
    print(f"Collecting: {film['title']}")
    posts_df = collect_pre_release_posts(film['title'], film['release_date'], days_before=30)
    all_posts[film['title']] = posts_df
    engagement_features[film['title']] = compute_engagement_features(posts_df)
    
print("Done!")


In [ ]:
# Preview engagement features for one film
import json
print(json.dumps(engagement_features.get('Oppenheimer', {}), indent=2))


In [ ]:
# Save raw posts
import os
os.makedirs('../data/raw', exist_ok=True)
for title, df in all_posts.items():
    safe_name = title.lower().replace(' ', '_')
    df.to_csv(f'../data/raw/{safe_name}_posts.csv', index=False)
print("Raw posts saved to data/raw/")


## Fetch box office data from The Numbers

The Numbers publishes public box office data. We scrape opening weekend and total gross per film.

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

def scrape_box_office(title: str) -> dict:
    """Simple scraper for The Numbers movie pages."""
    query = title.lower().replace(' ', '-').replace(':', '')
    url = f"https://www.the-numbers.com/movie/{query}"
    headers = {'User-Agent': 'Mozilla/5.0 (research project)'}
    
    try:
        r = requests.get(url, headers=headers, timeout=10)
        if r.status_code != 200:
            return {'title': title, 'error': f'HTTP {r.status_code}'}
        
        soup = BeautifulSoup(r.content, 'html.parser')
        # Parse opening weekend and total gross from the summary table
        # Note: exact selectors depend on current site structure
        tables = soup.find_all('table', {'id': 'movie_finances'})
        # ... parsing logic here ...
        return {'title': title, 'url': url, 'status': 'fetched'}
    except Exception as e:
        return {'title': title, 'error': str(e)}

# Test
result = scrape_box_office('Oppenheimer')
print(result)
time.sleep(2)  # Be polite to the server
